In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt

import numpy as np
import os

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q1_data.csv")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(6,4))
plt.plot(df["Delivery_Time"])
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop("Order_ID", inplace=True, axis = 1)

In [ ]:
# Task 2: Write your code here:
missing = df.isnull().sum() / len(df) * 100

missing_data = pd.DataFrame({"Columns": missing.index, 'Missing_Percentage': missing.values})

missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

missing_data

df = df.dropna(inplace=False)

In [ ]:
# Task 3: Write your code here:
print(df.duplicated().sum())
df.drop_duplicates(inplace = True)
print(df.duplicated().sum())

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
categories = df.select_dtypes(include=["object"]).columns

for col in categories:
  df[col] = le.fit_transform(df[col])

df.head()

In [ ]:
# Task 5: Write your code here:
# delivery distance, restaurant preparation time, traffic conditions, and order details.

from sklearn.preprocessing import StandardScaler
feature = ["Distance_km", "Preparation_Time_min", "Traffic_Level", "Weather", "Time_of_Day", "Vehicle_Type", "Courier_Experience_yrs"]
standard_scaler = StandardScaler()
df[feature] = standard_scaler.fit_transform(df[feature])

df.head()

In [ ]:
# Task 6: Write your code here:
#No need because it is a continous target and we are forming a linear regression model

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis = 1, inplace = False)
y = df["Delivery_Time"]

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []
mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mse_scores.append(mean_squared_error(y_test, y_pred))
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Step 4 :  print

# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"MSE : {np.mean(mse_scores):.2f}")
print(f"MAE : {np.mean(mae_scores):.2f}")
print(f"RMSE: {np.sqrt(np.mean(mse_scores)):.2f}")
print("-"*40)


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

y_pred = model.predict(X_test)

plt.figure(figsize=(6,4))
plt.hist(y_pred)
plt.show()



In [ ]:
%pip install kagglehub catboost lightgbm tqdm -q

In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

all_results = {}

for name in models:
  all_results[name] = {'mae': []}


kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/5")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)


    # Store results
    all_results[model_name]["mae"].append(mae)

for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")
